In [1]:
from pyspark.sql import SparkSession

import pyspark.sql.functions as F
from pyspark.sql.window import Window
import pyspark.sql.types as T

from datetime import datetime
from datetime import timedelta

import pandas as pd

from IPython.core.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

/tmp/ipykernel_9477/1599433937.py:12: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [2]:
spark = SparkSession.builder \
    .appName("create fix features") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .config("spark.cores.max", "12")\
    .config('spark.executor.memory', '4g')\
    .enableHiveSupport() \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 16:35:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
business_month_start = datetime(datetime.today().year, datetime.today().month, 1).date()
business_month_end = datetime(datetime.today().year, datetime.today().month + 1, 1).date() - timedelta(days = 1)
business_month_start, business_month_end

(datetime.date(2026, 3, 1), datetime.date(2026, 3, 31))

In [4]:
def fix_conv(spark, business_month_end):
    import pyspark.sql.functions as F
    from pyspark.sql.window import Window

    red_convergent = spark.table('mtsfix_dm.agg_mtsfix_red_convergent_fixa_a')
    mgts_convergent = spark.table('mtsfix_dm.agg_mtsfix_red_convergent_mgts_fixa_a')

    all_convergent_base = (
        red_convergent
        .unionAll(mgts_convergent)
        .where(F.col('dt_from') <= business_month_end)
        .where(F.coalesce(F.col('dt_to'), F.lit('9999-12-31')) > business_month_end)
        .select(
                'foris_id', 
                'foris_acc_id', 
                'foris_msisdn', 
                'fix_acc_num', 
                'fix_acc_id', 
                'personal_account_number', 
                'dt_from', 
                'foris_tariff_plan', 
                'fix_billing_id'
        )
    )

    foris_identity = (
        all_convergent_base
        .select(
                'foris_id', 
                'foris_acc_id', 
                'personal_account_number', 
                F.col('foris_id').alias('billing_id'),
                F.col('foris_acc_id').alias('acc_id'),
                F.col('personal_account_number').alias('acc_num'),
                'foris_msisdn', 
                'foris_tariff_plan', 
                F.lit(1).alias('is_foris')
        )
        .drop_duplicates()
    )

    fix_identity = (
        all_convergent_base
        .select(
                'foris_id', 
                'foris_acc_id', 
                'personal_account_number', 
                F.col('fix_billing_id').alias('billing_id'),
                F.col('fix_acc_id').alias('acc_id'),
                F.col('fix_acc_num').alias('acc_num'),
                'foris_msisdn', 
                'foris_tariff_plan', 
                F.lit(0).alias('is_foris')
        )
        .drop_duplicates()
    )

    foris_and_fix_identities = (foris_identity.unionAll(fix_identity))

    acc_con = spark.table('mtsfix_dm.agg_mtsfix_acc_con_fixa_a')

    adding_acc_con = (
        foris_and_fix_identities
        .join(
            acc_con
            .withColumn(
                'row_num',
                F.row_number().over(Window.partitionBy('billing_id', 'acc_id').orderBy(F.desc('dt_from')))
            )
            .where(F.col('row_num') == 1)
            .drop('row_num')
            .select(
                'acc_id',
                'bopos_id',
                'contract_id',
                'contract_num',
                'siebel_contract_id',
                'customer_id',
                'city_id',
                'reg_id',
                'mr_id',
                'customer_type_id',
                'calculation_method_id',
                'dt_from',
                'is_noncommercial',
                'is_closed',
                'parent_contract_id',
                'marketing_category_id',
                'siebel_status_id',
                'service_provider_id',
                'is_b2b',
                'orders_credit',
                'credit',
                'is_archive',
                'customer_category_id',
                'billing_id'
            ),
            ['billing_id', 'acc_id'],
            'left'
        )
    )
    return adding_acc_con

In [5]:
def fix_add_att(spark, business_month_end):
    import pyspark.sql.functions as F
    from pyspark.sql.window import Window

    acc_att = spark.table('mtsfix_dm.agg_mtsfix_acc_att_fixa_a')

    acc_att_processed = (
        acc_att
        .select(
            'billing_id',
            'acc_id',
            'dt_open',
            'dt_close',
            'dt_open_spd',
            'dt_close_spd',
            'dt_open_ctv',
            'dt_close_ctv',
            'dt_open_atv',
            'dt_close_atv',
            'dt_open_tlf',
            'dt_close_tlf',
            'dt_open_ictv',
            'dt_close_ictv'
        )
        .withColumn(
            'dt_open',
            F.when(
                (F.col('dt_open') < '2000-01-01') | (F.col('dt_open').isNull()),
                F.least(
                    F.to_date(F.col('dt_open_spd')), F.to_date(F.col('dt_open_ctv')), 
                    F.to_date(F.col('dt_open_atv')), 
                    F.to_date(F.col('dt_open_tlf')), F.to_date(F.col('dt_open_ictv'))
                )
            ).otherwise(F.to_date(F.col('dt_open')))
        )
        .where(F.col('dt_open').isNotNull())
        .withColumn('is_acc_att', F.lit(1))
        .withColumn(
            'is_contract_closed_flg', 
            F.when(F.to_date(F.col('dt_close')) <= F.lit(business_month_end), F.lit(1)).otherwise(F.lit(0))           
        )
        
        .withColumn(
            'live_dur_contract',
            F.when((F.col('dt_close') <= F.lit(business_month_end)) & F.col('dt_close').isNotNull(),
                F.lit(F.datediff(F.to_date(F.col('dt_close')), F.col('dt_open')))      
            ).otherwise(F.lit(F.datediff(F.lit(business_month_end), F.col('dt_open'))))
        )

        .withColumn(
            'live_dur_fix_spd',
            F.when((F.col('dt_close_spd') <= F.lit(business_month_end)) & F.col('dt_close_spd').isNotNull(),
                F.lit(F.datediff(F.to_date(F.col('dt_close_spd')), F.col('dt_open_spd')))      
            ).otherwise(F.lit(F.datediff(F.lit(business_month_end), F.col('dt_open_spd'))))
        )

        .withColumn(
            'live_dur_fix_telef',
            F.when((F.col('dt_close_tlf') <= F.lit(business_month_end)) & F.col('dt_close_tlf').isNotNull(),
                F.lit(F.datediff(F.to_date(F.col('dt_close_tlf')), F.col('dt_open_tlf')))      
            ).otherwise(F.lit(F.datediff(F.lit(business_month_end), F.col('dt_open_tlf'))))
        )

        .withColumn(
            'live_dur_fix_atv',
            F.when((F.col('dt_close_atv') <= F.lit(business_month_end)) & F.col('dt_close_atv').isNotNull(),
                F.lit(F.datediff(F.to_date(F.col('dt_close_atv')), F.col('dt_open_atv')))      
            ).otherwise(F.lit(F.datediff(F.lit(business_month_end), F.col('dt_open_atv'))))
        )

        .withColumn(
            'live_dur_fix_ctv',
            F.when((F.col('dt_close_ctv') <= F.lit(business_month_end)) & F.col('dt_close_ctv').isNotNull(),
                F.lit(F.datediff(F.to_date(F.col('dt_close_ctv')), F.col('dt_open_ctv')))      
            ).otherwise(F.lit(F.datediff(F.lit(business_month_end), F.col('dt_open_ctv'))))
        )

        .withColumn(
            'live_dur_fix_ictv',
            F.when((F.col('dt_close_ictv') <= F.lit(business_month_end)) & F.col('dt_close_ictv').isNotNull(),
                F.lit(F.datediff(F.to_date(F.col('dt_close_ictv')), F.col('dt_open_ictv')))      
            ).otherwise(F.lit(F.datediff(F.lit(business_month_end), F.col('dt_open_ictv'))))
        )

        .withColumn(
            'has_opened_spd',
            F.when(((F.col('dt_close_spd') > F.lit(business_month_end))
                   | (~F.col('dt_close_spd').isNotNull()))
                  & (F.col('dt_open_spd') <= F.lit(business_month_end)), 1).otherwise(0)
        )

        .withColumn(
            'has_opened_ctv',
            F.when(((F.col('dt_close_ctv') > F.lit(business_month_end))
                   | (~F.col('dt_close_ctv').isNotNull()))
                  & (F.col('dt_open_ctv') <= F.lit(business_month_end)), 1).otherwise(0)
        )

        .withColumn(
            'has_opened_atv',
            F.when(((F.col('dt_close_atv') > F.lit(business_month_end))
                   | (~F.col('dt_close_atv').isNotNull()))
                  & (F.col('dt_open_atv') <= F.lit(business_month_end)), 1).otherwise(0)
        )

        .withColumn(
            'has_opened_tlf',
            F.when(((F.col('dt_close_tlf') > F.lit(business_month_end))
                   | (~F.col('dt_close_tlf').isNotNull()))
                  & (F.col('dt_open_tlf') <= F.lit(business_month_end)), 1).otherwise(0)
        )

        .withColumn(
            'has_opened_ictv',
            F.when(((F.col('dt_close_ictv') > F.lit(business_month_end))
                   | (~F.col('dt_close_ictv').isNotNull()))
                  & (F.col('dt_open_ictv') <= F.lit(business_month_end)), 1).otherwise(0)
        )

        .drop(
            'dt_close',
            'dt_close_spd',
            'dt_close_ctv',
            'dt_close_atv',
            'dt_close_tlf',
            'dt_close_ictv'
        )
    )

    return acc_att_processed

In [6]:
def fix_add_pay(spark, business_month_end):
    import pyspark.sql.functions as F
    from pyspark.sql.window import Window

    dt_begin_fix_pay = pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(5)

    m0 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(0)).date()
    m1 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(1)).date()
    m2 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(2)).date()
    m3 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(3)).date()
    m4 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(4)).date()

    print(f'dt_begin_fix_pay: {dt_begin_fix_pay}')
    print(f'm0: {m0}')
    print(f'm1: {m1}')
    print(f'm2: {m2}')
    print(f'm3: {m3}')
    print(f'm4: {m4}')

    fix_pay = spark.table('mtsfix_dm.agg_mtsfix_pay_fixa_a')

    fix_pay_processed = (
        fix_pay
        .withColumnRenamed('dt', 'pay_date')
        
        .where(F.col('pay_date') > F.lit(dt_begin_fix_pay))
        .where(F.col('pay_date') <= F.lit(m0))
        .withColumn('dt', F.last_day(F.to_date('pay_date')))
        .select('billing_id', 'acc_id', 'dt', 'pay_class', 'amount')
    )

    pay_class = (
        fix_pay_processed
        .withColumn(
            'sum_class_mm',
            F.sum('amount').over(Window.partitionBy('billing_id', 'acc_id', 'dt', 'pay_class'))
        )

        .withColumn(
            'rnk',
            F.row_number().over(
                Window.partitionBy('billing_id', 'acc_id', 'dt').orderBy(F.col('sum_class_mm').desc()))
        )
        .where(F.col('rnk') == 1)
        .withColumn(
            'pay_class_00mm',
            F.when(F.col('dt') == F.lit(m0), F.col('pay_class')).otherwise(F.lit(0))
        )

        .withColumn(
            'pay_class_01mm',
            F.when(F.col('dt') == F.lit(m1), F.col('pay_class')).otherwise(F.lit(0))
        )

        .withColumn(
            'pay_class_02mm',
            F.when(F.col('dt') == F.lit(m2), F.col('pay_class')).otherwise(F.lit(0))
        )

        .withColumn(
            'pay_class_03mm',
            F.when(F.col('dt') == F.lit(m3), F.col('pay_class')).otherwise(F.lit(0))
        )

        .withColumn(
            'pay_class_04mm',
            F.when(F.col('dt') == F.lit(m4), F.col('pay_class')).otherwise(F.lit(0))
        )

        .groupBy('billing_id', 'acc_id')
        .agg(
            F.max('pay_class_00mm').alias('pay_class_00mm'),
            F.max('pay_class_01mm').alias('pay_class_01mm'),
            F.max('pay_class_02mm').alias('pay_class_02mm'),
            F.max('pay_class_03mm').alias('pay_class_03mm'),
            F.max('pay_class_04mm').alias('pay_class_04mm')
        )
    )

    pay_month = (
        fix_pay_processed
        .withColumn('pay_00mm', F.when(F.col('dt') == F.lit(m0), F.col('amount')).otherwise(F.lit(0)))
        .withColumn('pay_01mm', F.when(F.col('dt') == F.lit(m1), F.col('amount')).otherwise(F.lit(0)))
        .withColumn('pay_02mm', F.when(F.col('dt') == F.lit(m2), F.col('amount')).otherwise(F.lit(0)))
        .withColumn('pay_03mm', F.when(F.col('dt') == F.lit(m3), F.col('amount')).otherwise(F.lit(0)))
        .withColumn('pay_04mm', F.when(F.col('dt') == F.lit(m4), F.col('amount')).otherwise(F.lit(0)))

        .groupBy('billing_id', 'acc_id')
        .agg(
            F.sum('pay_00mm').alias('pay_00mm'),
            F.sum('pay_01mm').alias('pay_01mm'),
            F.sum('pay_02mm').alias('pay_02mm'),
            F.sum('pay_03mm').alias('pay_03mm'),
            F.sum('pay_04mm').alias('pay_04mm')
        )
        .withColumn('is_pay', F.lit(1))
    )

    result = (pay_month.join(pay_class, ['billing_id', 'acc_id'], 'left'))

    return result

In [7]:
def fix_add_traffic(spark, business_month_end):
    import pyspark.sql.functions as F
    from pyspark.sql.window import Window

    dt_begin_fix_traffic = pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(5)

    m0 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(0)).date()
    m1 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(1)).date()
    m2 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(2)).date()
    m3 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(3)).date()
    m4 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(4)).date()

    print(f'dt_begin_fix_traffic: {dt_begin_fix_traffic}')
    print(f'm0: {m0}')
    print(f'm1: {m1}')
    print(f'm2: {m2}')
    print(f'm3: {m3}')
    print(f'm4: {m4}')

    fix_clc = spark.table('mtsfix_dm_sec.agg_mtsfix_clc_fixa_a')

    fix_trafic = (
        fix_clc
        .where((F.col('dt') > dt_begin_fix_traffic) & (F.col('dt') <= business_month_end)
        )

        .withColumn('dt', F.col('dt').cast('date'))
        .withColumn(
            'gb',
            F.round(
                F.when(F.col('metric_unit_number') == 2,
                       F.col('number_of_units') / 8 / 1024 / 1024 / 1024)   
                      .when(F.col('metric_unit_number') == 3,
                           F.col('number_of_units') / 1024 / 1024 / 1024)
                      .when(F.col('metric_unit_number') == 4,
                           F.col('number_of_units') / 1024 / 1024)
                      .when(F.col('metric_unit_number') == 5,
                           F.col('number_of_units') / 1024).otherwise(F.lit(0)), 2
                )
        )
        .groupBy('billing_id', 'acc_id')
        .agg(
            F.sum(
                F.when(F.col('dt') == F.lit(m0), F.col('gb')).otherwise(F.lit(0))
            ).alias('shpd_gb_0'),
            F.sum(
                F.when(F.col('dt') == F.lit(m1), F.col('gb')).otherwise(F.lit(0))
            ).alias('shpd_gb_1'),
            F.sum(
                F.when(F.col('dt') == F.lit(m2), F.col('gb')).otherwise(F.lit(0))
            ).alias('shpd_gb_2'),
            F.sum(
                F.when(F.col('dt') == F.lit(m3), F.col('gb')).otherwise(F.lit(0))
            ).alias('shpd_gb_3'),
            F.sum(
                F.when(F.col('dt') == F.lit(m4), F.col('gb')).otherwise(F.lit(0))
            ).alias('shpd_gb_4')
        )
        .withColumn('is_traffic', F.lit(1))
    )

    return fix_trafic

In [8]:
def fix_add_equip(spark, business_month_end):
    import pyspark.sql.functions as F
    from pyspark.sql.window import Window
    import itertools

    equipment_types = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
    provide_types = [1, 2, 3, 4]
    categs = list(itertools.product(equipment_types, provide_types))

    names_categ = [f'equipment_type_{str(cat[0])}_provide_type_{str(cat[1])}' for cat in categs]

    exprs = [
        F.when(
            (F.col('equipment_type') == cat[0]) & (F.col('provide_type') == cat[1]), 1
        ).otherwise(0).alias(f'equipment_type_{str(cat[0])}_provide_type_{str(cat[1])}') for cat in categs
    ]

    fix_eqipment = spark.table('mtsfix_dm.agg_mtsfix_equipment_fixa_a')

    fix_eqipment_processed = (
        fix_eqipment
        .where(F.col('dt_from') <= F.lit(business_month_end))
        .where(F.coalesce(F.col('dt_to'), F.lit('9999-12-31')) > F.lit(business_month_end))
        .select(exprs + ['acc_id', 'billing_id', 'equipment_type', 'provide_type'])
        .groupBy('billing_id', 'acc_id')
        .max(*names_categ)
        .withColumn('is_equip', F.lit(1))
    )

    fix_old_eq = (
        fix_eqipment
        .select('billing_id', 'acc_id')
        .distinct()
        .withColumn('had_old_eq', F.lit(1))
    )

    fix_block_eq = (
        fix_eqipment
        .where(F.col('is_blocked') == 1)
        .select('billing_id', 'acc_id')
        .distinct()
        .withColumn('had_block_eq', F.lit(1))
    )

    for column in fix_eqipment_processed.drop('billing_id', 'acc_id', 'is_equip').columns:
        start_index = column.find('(')
        end_index = column.find(')')

        if start_index and end_index:
            fix_eqipment_processed = (
                fix_eqipment_processed
                .withColumnRenamed(column, column[start_index + 1: end_index])
            )
    result = (
        fix_eqipment_processed
        .join(fix_old_eq, ['billing_id', 'acc_id'], 'left')
        .join(fix_block_eq, ['billing_id', 'acc_id'], 'left')
    )

    return result

In [9]:
def fix_add_remedy(spark, business_month_end, adding_fix_equipment):
    import pyspark.sql.functions as F
    from pyspark.sql.window import Window

    dt_begin_remedytb = pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(5)

    m0 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(0)).date()
    m1 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(1)).date()
    m2 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(2)).date()
    m3 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(3)).date()
    m4 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(4)).date()

    print(f'dt_begin_fix_remedy: {dt_begin_remedytb}')
    print(f'm0: {m0}')
    print(f'm1: {m1}')
    print(f'm2: {m2}')
    print(f'm3: {m3}')
    print(f'm4: {m4}')

    remedytb_processed = (
        spark.table('raw.mtsru_remedytb_msk__aradmin_stb_singlinc_dashb_all')
        .where(F.col('raw_dt') > F.lit(dt_begin_remedytb))
        .where(F.col('dogtype') == 'Фиксированный')
        .where(F.col('createdate_dt_orig') > F.lit(dt_begin_remedytb))
        .where(F.col('createdate_dt_orig') <= F.lit(business_month_end))
        .where(F.col('request_id').isNotNull())
        .select(
            "request_id",
            "status",
            "priority",
            "contract_num",
            "closetime_dt_orig",
            "createdate_dt_orig"
        )
        .drop_duplicates()
    )

    remedytb_processed_grouped_request = (
        remedytb_processed
        .groupBy('request_id')
        .agg(
            F.count_distinct('contract_num').alias('cnt_con')
        )
        .where(F.col('cnt_con') == 1)
    )

    remedytb_agg = (
        remedytb_processed
        .join(remedytb_processed_grouped_request, ['request_id'], 'inner')
        .select(
            'request_id',
            'status',
            'priority',
            'contract_num',
            'closetime_dt_orig',
            'createdate_dt_orig',
            F.last_day(F.to_date(F.col('createdate_dt_orig'))).alias('create_dt')
        )
    )

    prefinal_remedytb_agg = (
        adding_fix_equipment
        .select('billing_id', 'acc_id', 'acc_num', 'contract_num')
        .drop_duplicates()
        .join(remedytb_agg, ['contract_num'], 'inner')
        .withColumn(
            'complaints_num_active_expec',
            F.when(F.col('status') == 'Активное ожидание', 1).otherwise(0)
        )
        .withColumn('complaints_num_active', F.when(F.col('status') == 'В работе', 1).otherwise(0))
        .withColumn('complaints_num_close', F.when(F.col('status') == 'Закрыт', 1).otherwise(0))
        
        .withColumn(
            'complaints_num_priority_low',
            F.when(F.col('priority') == 'Низкий', 1).otherwise(0)
        )
        
        .withColumn(
            'complaints_num_priority_mid',
            F.when(F.col('priority') == 'Средний', 1).otherwise(0)
        )

        .withColumn(
            'complaints_num_priority_lowmid',
            F.when(F.col('priority') == 'Ниже среднего', 1).otherwise(0)
        )

        .withColumn(
            'complaints_num_priority_high',
            F.when(F.col('priority') == 'Высокий', 1).otherwise(0)
        )

        .withColumn(
            'complaints_num_priority_highmid',
            F.when(F.col('priority') == 'Выше среднего', 1).otherwise(0)
        )

        .withColumn(
            'complaints_num_priority_highest',
            F.when(F.col('priority') == 'Наивысший', 1).otherwise(0)
        )

        .withColumn(
            'zhalobi_lifetime_hours',
            F.when(F.col('status').isin(['Закрыт', 'Решен']),
                    (F.col('closetime_dt_orig').cast('long') - F.col('createdate_dt_orig').cast('long')) / 3600
                  ).otherwise(0)
            
        )

        .withColumn(
            'cnt_zhalobi_00mm',
            F.when(F.col('create_dt') == F.lit(m0), 1).otherwise(F.lit(0))
        )
        .withColumn(
            'cnt_zhalobi_01mm',
            F.when(F.col('create_dt') == F.lit(m1), 1).otherwise(F.lit(0))
        )
        .withColumn(
            'cnt_zhalobi_02mm',
            F.when(F.col('create_dt') == F.lit(m2), 1).otherwise(F.lit(0))
        )
        .withColumn(
            'cnt_zhalobi_03mm',
            F.when(F.col('create_dt') == F.lit(m3), 1).otherwise(F.lit(0))
        )
        .withColumn(
            'cnt_zhalobi_04mm',
            F.when(F.col('create_dt') == F.lit(m4), 1).otherwise(F.lit(0))
        )

        .groupBy('billing_id', 'acc_id', 'acc_num', 'contract_num', 'request_id')
        .agg(
            F.max('complaints_num_active_expec').alias('complaints_num_active_expec'),
            F.max('complaints_num_active').alias('complaints_num_active'),
            F.max('complaints_num_close').alias('complaints_num_close'),
            F.max('complaints_num_priority_low').alias('complaints_num_priority_low'),
            F.max('complaints_num_priority_mid').alias('complaints_num_priority_mid'),
            F.max('complaints_num_priority_lowmid').alias('complaints_num_priority_lowmid'),
            F.max('complaints_num_priority_high').alias('complaints_num_priority_high'),
            F.max('complaints_num_priority_highmid').alias('complaints_num_priority_highmid'),
            F.max('complaints_num_priority_highest').alias('complaints_num_priority_highest'),
            F.max('zhalobi_lifetime_hours').alias('zhalobi_lifetime_hours'),
            F.max('cnt_zhalobi_00mm').alias('cnt_zhalobi_00mm'),
            F.max('cnt_zhalobi_01mm').alias('cnt_zhalobi_01mm'),
            F.max('cnt_zhalobi_02mm').alias('cnt_zhalobi_02mm'),
            F.max('cnt_zhalobi_03mm').alias('cnt_zhalobi_03mm'),
            F.max('cnt_zhalobi_04mm').alias('cnt_zhalobi_04mm')
        )
    )

    final_remedytb_agg = (
        prefinal_remedytb_agg
        .groupBy('billing_id', 'acc_id', 'acc_num', 'contract_num')
        .agg(
            F.count_distinct('request_id').alias('cnt_zhalobi'),
            
            F.sum('cnt_zhalobi_00mm').alias('cnt_zhalobi_00mm'),
            F.sum('cnt_zhalobi_01mm').alias('cnt_zhalobi_01mm'),
            F.sum('cnt_zhalobi_02mm').alias('cnt_zhalobi_02mm'),
            F.sum('cnt_zhalobi_03mm').alias('cnt_zhalobi_03mm'),
            F.sum('cnt_zhalobi_04mm').alias('cnt_zhalobi_04mm'),
            
            F.min('zhalobi_lifetime_hours').alias('min_zhalobi_lifetime_hours'),
            F.max('zhalobi_lifetime_hours').alias('max_zhalobi_lifetime_hours'),
            
            F.sum('complaints_num_active_expec').alias('complaints_num_active_expec'),
            F.sum('complaints_num_active').alias('complaints_num_active'),
            F.sum('complaints_num_close').alias('complaints_num_close'),
            F.sum('complaints_num_priority_low').alias('complaints_num_priority_low'),
            F.sum('complaints_num_priority_mid').alias('complaints_num_priority_mid'),
            F.sum('complaints_num_priority_lowmid').alias('complaints_num_priority_lowmid'),
            F.sum('complaints_num_priority_high').alias('complaints_num_priority_high'),
            F.sum('complaints_num_priority_highmid').alias('complaints_num_priority_highmid'),
            F.sum('complaints_num_priority_highest').alias('complaints_num_priority_highest')
        )
        .withColumn('is_zhalobi', F.lit(1))
    )

    return final_remedytb_agg

In [10]:
def fix_add_serv(spark, business_month_end):
    import pyspark.sql.functions as F
    from pyspark.sql.window import Window

    fix_services = spark.table('mtsfix_dm.agg_mtsfix_app_service_fixa_a')
    
    services = (
        fix_services
        .where(F.col('dt_from') <= F.current_date())
        .where(F.col('dt_to') > F.current_date())
        .where(F.col('is_closed') == 0)
    )

    count_closed = (
        fix_services
        .select('billing_id', 'acc_id', 'app_service_id', 'is_closed')
        .distinct()
        .groupBy('billing_id', 'acc_id')
        .agg(
            F.sum('is_closed').alias('closed_cnt')
        )
    )

    count_basic = (
        services
        .select('billing_id', 'acc_id', 'app_service_id', 'is_b2c_basic_service')
        .distinct()
        .groupBy('billing_id', 'acc_id')
        .agg(
            F.count('is_b2c_basic_service').alias('all_serv_cnt'),
            F.sum('is_b2c_basic_service').alias('basic_serv_cnt')
        )
        .withColumn('not_basic_serv_cnt', F.col('all_serv_cnt') - F.col('basic_serv_cnt'))
        .drop('all_serv_cnt')
    )

    detail_serv = (
        services
        .groupBy('billing_id', 'acc_id')
        .agg(
            F.count_distinct('app_service_id').alias('active_serv_cnt')
        )
        
        .join(count_closed, ['billing_id', 'acc_id'], 'left')
        .join(count_basic, ['billing_id', 'acc_id'], 'left')
    )

    disc = spark.table('mtsfix_dm_sec.agg_mtsfix_discount_fixa_a')

    disc_processed = (
        disc
        .where(F.col('dt_from') <= F.current_date())
        .where(F.col('dt_to') > F.current_date())
    )

    acc_disc = (
        services
        .join(disc_processed, ['billing_id', 'acc_id', 'app_service_id'], 'left')

        .groupBy('billing_id', 'acc_id')
        .agg(
            F.sum('discount_price').alias('all_disc_sum')
        )
    )

    result = (
        detail_serv
        .join(acc_disc, ['billing_id', 'acc_id'], 'left')
    )

    return result

In [11]:
def fix_add_gr(spark):
    import pyspark.sql.functions as F
    from pyspark.sql.window import Window

    gr_features = spark.table('gr_dm.ma_fix_acc_services_history')

    gr_features_fix = (
        gr_features
        .select(
            'billing_id',
            'customer_id',
            'version_dt',
            'bb_rent_amount',
            'tv_rent_amount',
            'tlf_rent_amount',
            'tv_line_coast',
            'tv_mult_coast',
            'gender',
            'customer_age',
            'sale_channel',
            'lk_use_count',
            'lk_use_count_3m',
            'all_sr_count_30d',
            'complaint_sr_count_30d',
            'cm_sr_count_30d',
            'cm_sr_count_180d'
        )

        .withColumn(
            'row_num',
            F.row_number().over(Window.partitionBy('billing_id', 'customer_id').orderBy(F.desc('version_dt')))
        )
        .where(F.col('row_num') == 1)
        .drop('version_dt', 'row_num')
        .distinct()
    )

    return gr_features_fix

In [12]:
def fix_final_df(union_df, business_month_end):
    import pyspark.sql.functions as F
    from pyspark.sql.window import Window

    final_df = (
        union_df
        .withColumn('billing_id', F.when(F.col('is_foris') == 0, F.col('billing_id')).otherwise(F.lit(None)))
        .withColumn('acc_id', F.when(F.col('is_foris') == 0, F.col('acc_id')).otherwise(F.lit(None)))
        .withColumn('acc_num', F.when(F.col('is_foris') == 0, F.col('acc_num')).otherwise(F.lit(None)))
        .withColumn(
            'foris_contract_num', 
            F.when(F.col('is_foris') == 1, F.col('contract_num')).otherwise(F.lit(None))
        )
        .withColumn(
            'total_live_dur', F.lit(F.datediff(F.to_date(F.lit(business_month_end)), F.col('dt_from')))
        )
        .groupBy('foris_id', 'foris_acc_id', 'personal_account_number', 'foris_msisdn', 'foris_tariff_plan')
        .agg(
            F.first('billing_id', ignorenulls = True).alias('billing_id'),
            F.first('acc_id', ignorenulls = True).alias('acc_id'),
            F.first('acc_num', ignorenulls = True).alias('acc_num'),
            F.first('foris_contract_num', ignorenulls = True).alias('foris_contract_num'),
            
            F.max('total_live_dur').alias('bopos_id'),
            F.first('is_noncommercial', ignorenulls = True).alias('is_noncommercial'),
            F.max('is_closed').alias('is_closed'),
            F.sum('orders_credit').alias('orders_credit'),
            F.sum('credit').alias('credit'),
            F.max('is_archive').alias('is_archive'),
            F.max('is_contract_closed_flg').alias('is_contract_closed_flg'),
            
            F.max('live_dur_contract').alias('live_dur_contract'),
            F.max('live_dur_fix_spd').alias('live_dur_fix_spd'),
            F.max('live_dur_fix_telef').alias('live_dur_fix_telef'),
            F.max('live_dur_fix_atv').alias('live_dur_fix_atv'),
            F.max('live_dur_fix_ctv').alias('live_dur_fix_ctv'),
            F.max('live_dur_fix_ictv').alias('live_dur_fix_ictv'),
            
            F.max('has_opened_spd').alias('has_opened_spd'),
            F.max('has_opened_ctv').alias('has_opened_ctv'),
            F.max('has_opened_atv').alias('has_opened_atv'),
            F.max('has_opened_tlf').alias('has_opened_tlf'),
            F.max('has_opened_ictv').alias('has_opened_ictv'),

            F.sum('pay_00mm').alias('pay_00mm'),
            F.sum('pay_01mm').alias('pay_01mm'),
            F.sum('pay_02mm').alias('pay_02mm'),
            F.sum('pay_03mm').alias('pay_03mm'),
            F.sum('pay_04mm').alias('pay_04mm'),

            F.sum('shpd_gb_0').alias('shpd_gb_0'),
            F.sum('shpd_gb_1').alias('shpd_gb_1'),
            F.sum('shpd_gb_2').alias('shpd_gb_2'),
            F.sum('shpd_gb_3').alias('shpd_gb_3'),
            F.sum('shpd_gb_4').alias('shpd_gb_4'),

            F.sum('cnt_zhalobi').alias('cnt_zhalobi'),

            F.sum('cnt_zhalobi_00mm').alias('cnt_zhalobi_00mm'),
            F.sum('cnt_zhalobi_01mm').alias('cnt_zhalobi_01mm'),
            F.sum('cnt_zhalobi_02mm').alias('cnt_zhalobi_02mm'),
            F.sum('cnt_zhalobi_03mm').alias('cnt_zhalobi_03mm'),
            F.sum('cnt_zhalobi_04mm').alias('cnt_zhalobi_04mm'),

            F.max('min_zhalobi_lifetime_hours').alias('min_zhalobi_lifetime_hours'),
            F.max('max_zhalobi_lifetime_hours').alias('max_zhalobi_lifetime_hours'),

            F.max('complaints_num_active_expec').alias('complaints_num_active_expec'),
            F.max('complaints_num_active').alias('complaints_num_active'),
            F.max('complaints_num_close').alias('complaints_num_close'),
            F.max('complaints_num_priority_low').alias('complaints_num_priority_low'),
            F.max('complaints_num_priority_mid').alias('complaints_num_priority_mid'),
            F.max('complaints_num_priority_lowmid').alias('complaints_num_priority_lowmid'),
            F.max('complaints_num_priority_high').alias('complaints_num_priority_high'),
            F.max('complaints_num_priority_highmid').alias('complaints_num_priority_highmid'),
            F.max('complaints_num_priority_highest').alias('complaints_num_priority_highest'),

            F.max('pay_class_00mm').alias('pay_class_00mm'),
            F.max('pay_class_01mm').alias('pay_class_01mm'),
            F.max('pay_class_02mm').alias('pay_class_02mm'),
            F.max('pay_class_03mm').alias('pay_class_03mm'),
            F.max('pay_class_04mm').alias('pay_class_04mm'),

            F.max('equipment_type_1_provide_type_1').alias('equipment_type_1_provide_type_1'),
            F.max('equipment_type_1_provide_type_2').alias('equipment_type_1_provide_type_2'),
            F.max('equipment_type_1_provide_type_3').alias('equipment_type_1_provide_type_3'),
            F.max('equipment_type_1_provide_type_4').alias('equipment_type_1_provide_type_4'),

            F.max('equipment_type_2_provide_type_1').alias('equipment_type_2_provide_type_1'),
            F.max('equipment_type_2_provide_type_2').alias('equipment_type_2_provide_type_2'),
            F.max('equipment_type_2_provide_type_3').alias('equipment_type_2_provide_type_3'),
            F.max('equipment_type_2_provide_type_4').alias('equipment_type_2_provide_type_4'),

            F.max('equipment_type_3_provide_type_1').alias('equipment_type_3_provide_type_1'),
            F.max('equipment_type_3_provide_type_2').alias('equipment_type_3_provide_type_2'),
            F.max('equipment_type_3_provide_type_3').alias('equipment_type_3_provide_type_3'),
            F.max('equipment_type_3_provide_type_4').alias('equipment_type_3_provide_type_4'),

            F.max('equipment_type_4_provide_type_1').alias('equipment_type_4_provide_type_1'),
            F.max('equipment_type_4_provide_type_2').alias('equipment_type_4_provide_type_2'),
            F.max('equipment_type_4_provide_type_3').alias('equipment_type_4_provide_type_3'),
            F.max('equipment_type_4_provide_type_4').alias('equipment_type_4_provide_type_4'),

            F.max('equipment_type_5_provide_type_1').alias('equipment_type_5_provide_type_1'),
            F.max('equipment_type_5_provide_type_2').alias('equipment_type_5_provide_type_2'),
            F.max('equipment_type_5_provide_type_3').alias('equipment_type_5_provide_type_3'),
            F.max('equipment_type_5_provide_type_4').alias('equipment_type_5_provide_type_4'),

            F.max('equipment_type_6_provide_type_1').alias('equipment_type_6_provide_type_1'),
            F.max('equipment_type_6_provide_type_2').alias('equipment_type_6_provide_type_2'),
            F.max('equipment_type_6_provide_type_3').alias('equipment_type_6_provide_type_3'),
            F.max('equipment_type_6_provide_type_4').alias('equipment_type_6_provide_type_4'),

            F.max('equipment_type_7_provide_type_1').alias('equipment_type_7_provide_type_1'),
            F.max('equipment_type_7_provide_type_2').alias('equipment_type_7_provide_type_2'),
            F.max('equipment_type_7_provide_type_3').alias('equipment_type_7_provide_type_3'),
            F.max('equipment_type_7_provide_type_4').alias('equipment_type_7_provide_type_4'),

            F.max('equipment_type_8_provide_type_1').alias('equipment_type_8_provide_type_1'),
            F.max('equipment_type_8_provide_type_2').alias('equipment_type_8_provide_type_2'),
            F.max('equipment_type_8_provide_type_3').alias('equipment_type_8_provide_type_3'),
            F.max('equipment_type_8_provide_type_4').alias('equipment_type_8_provide_type_4'),

            F.max('equipment_type_9_provide_type_1').alias('equipment_type_9_provide_type_1'),
            F.max('equipment_type_9_provide_type_2').alias('equipment_type_9_provide_type_2'),
            F.max('equipment_type_9_provide_type_3').alias('equipment_type_9_provide_type_3'),
            F.max('equipment_type_9_provide_type_4').alias('equipment_type_9_provide_type_4'),

            F.max('equipment_type_10_provide_type_1').alias('equipment_type_10_provide_type_1'),
            F.max('equipment_type_10_provide_type_2').alias('equipment_type_10_provide_type_2'),
            F.max('equipment_type_10_provide_type_3').alias('equipment_type_10_provide_type_3'),
            F.max('equipment_type_10_provide_type_4').alias('equipment_type_10_provide_type_4'),

            F.max('equipment_type_17_provide_type_1').alias('equipment_type_17_provide_type_1'),
            F.max('equipment_type_17_provide_type_2').alias('equipment_type_17_provide_type_2'),
            F.max('equipment_type_17_provide_type_3').alias('equipment_type_17_provide_type_3'),
            F.max('equipment_type_17_provide_type_4').alias('equipment_type_17_provide_type_4'),

            F.max('equipment_type_18_provide_type_1').alias('equipment_type_18_provide_type_1'),
            F.max('equipment_type_18_provide_type_2').alias('equipment_type_18_provide_type_2'),
            F.max('equipment_type_18_provide_type_3').alias('equipment_type_18_provide_type_3'),
            F.max('equipment_type_18_provide_type_4').alias('equipment_type_18_provide_type_4'),

            F.max('equipment_type_19_provide_type_1').alias('equipment_type_19_provide_type_1'),
            F.max('equipment_type_19_provide_type_2').alias('equipment_type_19_provide_type_2'),
            F.max('equipment_type_19_provide_type_3').alias('equipment_type_19_provide_type_3'),
            F.max('equipment_type_19_provide_type_4').alias('equipment_type_19_provide_type_4'),

            F.max('had_old_eq').alias('had_old_eq'),
            F.max('had_block_eq').alias('had_block_eq'),

            F.sum('active_serv_cnt').alias('active_serv_cnt'),
            F.sum('closed_cnt').alias('closed_cnt'),
            F.sum('basic_serv_cnt').alias('basic_serv_cnt'),
            F.sum('not_basic_serv_cnt').alias('not_basic_serv_cnt'),
            F.sum('all_disc_sum').alias('all_disc_sum'),

            F.max('bb_rent_amount').alias('bb_rent_amount'),
            F.max('tv_rent_amount').alias('tv_rent_amount'),
            F.max('tlf_rent_amount').alias('tlf_rent_amount'),
            F.max('tv_line_coast').alias('tv_line_coast'),
            F.max('tv_mult_coast').alias('tv_mult_coast'),

            F.first('gender', ignorenulls = True).alias('gender'),
            F.max('customer_age').alias('customer_age'),
            F.first('sale_channel', ignorenulls = True).alias('sale_channel'),

            F.sum('lk_use_count').alias('lk_use_count'),
            F.sum('all_sr_count_30d').alias('all_sr_count_30d'),
            F.sum('complaint_sr_count_30d').alias('complaint_sr_count_30d'),
            F.sum('cm_sr_count_30d').alias('cm_sr_count_30d'),
            F.sum('cm_sr_count_180d').alias('cm_sr_count_180d'),
            F.sum('lk_use_count_3m').alias('lk_use_count_3m')
        )
    )

    return final_df

In [13]:
conv = fix_conv(spark, business_month_end)
att = fix_add_att(spark, business_month_end)
pay = fix_add_pay(spark, business_month_end)
traffic = fix_add_traffic(spark, business_month_end)
equip = fix_add_equip(spark, business_month_end)
remedy = fix_add_remedy(spark, business_month_end, conv)
serv = fix_add_serv(spark, business_month_end)
gr = fix_add_gr(spark)

dt_begin_fix_pay: 2025-10-31 00:00:00
m0: 2026-03-31
m1: 2026-02-28
m2: 2026-01-31
m3: 2025-12-31
m4: 2025-11-30
dt_begin_fix_traffic: 2025-10-31 00:00:00
m0: 2026-03-31
m1: 2026-02-28
m2: 2026-01-31
m3: 2025-12-31
m4: 2025-11-30
dt_begin_fix_remedy: 2025-10-31 00:00:00
m0: 2026-03-31
m1: 2026-02-28
m2: 2026-01-31
m3: 2025-12-31
m4: 2025-11-30


In [15]:
union_df = (
    conv
    .join(att, ['billing_id', 'acc_id'], 'left')
    .join(pay, ['billing_id', 'acc_id'], 'left')
    .join(traffic, ['billing_id', 'acc_id'], 'left')
    .join(equip, ['billing_id', 'acc_id'], 'left')
    .join(remedy, ['billing_id', 'acc_id', 'acc_num', 'contract_num'], 'left')
    .join(serv, ['billing_id', 'acc_id'], 'left')
    .join(gr, ['billing_id', 'customer_id'], 'left')
)

In [16]:
final_df = fix_final_df(union_df, business_month_end)

In [17]:
(
    final_df
    .withColumn('table_business_month', F.lit(business_month_end))
    .coalesce(1)
    .write
    .partitionBy('table_business_month')
    .format('orc')
    .mode('overwrite')
    .saveAsTable('spfix_dm.fix_features_convergent')
)

26/03/01 16:35:27 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/01 16:37:59 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


In [18]:
spark.table('spfix_dm.fix_features_convergent').count()

2510247

In [20]:
spark.table('spfix_dm.fix_features_convergent').show(10, truncate = False)

+--------+------------+-----------------------+------------+-----------------+----------+------+-------+------------------+--------+----------------+---------+-------------+------+----------+----------------------+-----------------+----------------+------------------+----------------+----------------+-----------------+--------------+--------------+--------------+--------------+---------------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+-----------+----------------+----------------+----------------+----------------+----------------+--------------------------+--------------------------+---------------------------+---------------------+--------------------+---------------------------+---------------------------+------------------------------+----------------------------+-------------------------------+-------------------------------+--------------+--------------+--------------+--------------+--------------+-------------------------------+

In [50]:
spark.catalog.refreshTable('raw.mtsru_remedytb_msk__aradmin_stb_singlinc_dashb_all')

In [36]:
fix_add_gr(spark).show(10)

[Stage 55:>                                                         (0 + 1) / 1]

+----------+-----------+--------------+--------------+---------------+-------------+-------------+------+------------+------------+------------+---------------+----------------+----------------------+---------------+----------------+
|billing_id|customer_id|bb_rent_amount|tv_rent_amount|tlf_rent_amount|tv_line_coast|tv_mult_coast|gender|customer_age|sale_channel|lk_use_count|lk_use_count_3m|all_sr_count_30d|complaint_sr_count_30d|cm_sr_count_30d|cm_sr_count_180d|
+----------+-----------+--------------+--------------+---------------+-------------+-------------+------+------------+------------+------------+---------------+----------------+----------------------+---------------+----------------+
|         0|        619|          1230|           901|            103|          181|          151|     1|          25|          43|          18|             78|              31|                    16|             25|             102|
|         0|        716|          1430|          1246|          

In [34]:
fix_add_serv(spark, business_month_end).where(F.col('all_disc_sum').isNotNull()).show(10)

[Stage 52:>                                                         (0 + 1) / 1]

+----------+------+---------------+----------+--------------+------------------+------------+
|billing_id|acc_id|active_serv_cnt|closed_cnt|basic_serv_cnt|not_basic_serv_cnt|all_disc_sum|
+----------+------+---------------+----------+--------------+------------------+------------+
|         0| 37946|              5|         1|             5|                 0|          47|
|         0|167653|              4|         3|             3|                 1|          72|
|         0|199540|              2|         3|             2|                 1|           9|
|         0|279643|              2|         0|             1|                 1|          12|
|         0|286489|              3|         2|             2|                 1|          80|
|         0|324267|              1|         6|             0|                 1|          77|
|         0|334035|              1|         3|             0|                 1|           1|
|         0|387712|              2|         1|             1

In [33]:
fix_add_serv(spark, business_month_end).show(10)

26/03/01 15:22:55 WARN TaskSetManager: Lost task 0.0 in stage 17.0 (TID 65) (172.18.0.7 executor 0): org.apache.spark.memory.SparkOutOfMemoryError: [UNABLE_TO_ACQUIRE_MEMORY] Unable to acquire 262144 bytes of memory, got 0.
	at org.apache.spark.errors.SparkCoreErrors$.outOfMemoryError(SparkCoreErrors.scala:467)
	at org.apache.spark.errors.SparkCoreErrors.outOfMemoryError(SparkCoreErrors.scala)
	at org.apache.spark.memory.MemoryConsumer.throwOom(MemoryConsumer.java:157)
	at org.apache.spark.memory.MemoryConsumer.allocateArray(MemoryConsumer.java:98)
	at org.apache.spark.unsafe.map.BytesToBytesMap.allocate(BytesToBytesMap.java:865)
	at org.apache.spark.unsafe.map.BytesToBytesMap.reset(BytesToBytesMap.java:969)
	at org.apache.spark.sql.execution.UnsafeKVExternalSorter.<init>(UnsafeKVExternalSorter.java:173)
	at org.apache.spark.sql.execution.UnsafeFixedWidthAggregationMap.destructAndCreateExternalSorter(UnsafeFixedWidthAggregationMap.java:243)
	at org.apache.spark.sql.catalyst.expressions

+----------+------+---------------+----------+--------------+------------------+------------+
|billing_id|acc_id|active_serv_cnt|closed_cnt|basic_serv_cnt|not_basic_serv_cnt|all_disc_sum|
+----------+------+---------------+----------+--------------+------------------+------------+
|         0|108867|              1|         1|             1|                 0|        null|
|         0|148589|              1|         3|             0|                 1|        null|
|         0|195027|              3|         2|             2|                 2|        null|
|         0|288671|              3|         3|             1|                 2|        null|
|         0|326265|              3|         2|             1|                 2|        null|
|         0|336398|              2|         2|             0|                 2|        null|
|         0|375980|              2|         1|             2|                 0|        null|
|         0|387881|              3|         3|             3

In [ ]:
fix_add_remedy(spark, business_month_end, adding_fix_equipment)

In [24]:
fix_add_equip(spark, business_month_end).show(10)

[Stage 14:>                                                         (0 + 1) / 1]

+----------+------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+---------------------

In [23]:
spark.catalog.refreshTable('mtsfix_dm.agg_mtsfix_equipment_fixa_a')

In [13]:
spark.catalog.refreshTable('mtsfix_dm_sec.agg_mtsfix_clc_fixa_a')

In [14]:
fix_add_traffic(spark, business_month_end).show(10)

dt_begin_fix_pay: 2025-10-31 00:00:00
m0: 2026-03-31
m1: 2026-02-28
m2: 2026-01-31
m3: 2025-12-31
m4: 2025-11-30


[Stage 4:===================================================>       (7 + 1) / 8]

+----------+------+---------+---------+---------+---------+---------+----------+
|billing_id|acc_id|shpd_gb_0|shpd_gb_1|shpd_gb_2|shpd_gb_3|shpd_gb_4|is_traffic|
+----------+------+---------+---------+---------+---------+---------+----------+
|         0|  2395|   206.33|    47.50|   501.18|   787.22|   832.65|         1|
|         0| 51836|    87.18|   394.27|   108.44|   922.81|   504.59|         1|
|         0| 81945|   326.18|   519.05|   612.00|   593.63|   908.17|         1|
|         0| 88649|   385.59|   510.16|   614.17|    94.24|   469.46|         1|
|         0|100636|   126.87|   407.20|   809.76|   146.87|   509.04|         1|
|         0|105074|   811.19|   286.83|   670.71|   434.26|   590.20|         1|
|         0|108867|   910.03|    48.82|   804.15|   131.82|   905.63|         1|
|         0|110957|   742.10|   891.80|   452.15|   361.02|   170.71|         1|
|         0|118545|   353.80|   345.83|   697.72|   735.25|   529.32|         1|
|         0|119548|   268.34

In [40]:
fix_add_traffic(spark, business_month_end).show(10)

dt_begin_fix_pay: 2025-10-31 00:00:00
m0: 2026-03-31
m1: 2026-02-28
m2: 2026-01-31
m3: 2025-12-31
m4: 2025-11-30


[Stage 0:=====================================================>   (15 + 1) / 16]

+----------+------+---------+---------+---------+---------+---------+----------+
|billing_id|acc_id|shpd_gb_0|shpd_gb_1|shpd_gb_2|shpd_gb_3|shpd_gb_4|is_traffic|
+----------+------+---------+---------+---------+---------+---------+----------+
|         0|  2395|     0.00|     0.00|     0.00|     0.00|     0.00|         1|
|         0| 51836|     0.00|     0.00|     0.00|     0.00|     0.00|         1|
|         0| 81945|     0.00|     0.00|     0.00|     0.00|     0.00|         1|
|         0| 88649|     0.00|     0.00|     0.00|     0.00|     0.00|         1|
|         0|100636|     0.00|     0.00|     0.00|     0.00|     0.00|         1|
|         0|105074|     0.00|     0.00|     0.00|     0.00|     0.00|         1|
|         0|108867|     0.00|     0.00|     0.00|     0.00|     0.00|         1|
|         0|110957|     0.00|     0.00|     0.00|     0.00|     0.00|         1|
|         0|118545|     0.00|     0.00|     0.00|     0.00|     0.00|         1|
|         0|119548|     0.00

In [31]:
fix_add_traffic(spark, business_month_end).where(F.col('shpd_gb_2') != 0).show(10)

dt_begin_fix_pay: 2025-10-31 00:00:00
m0: 2026-03-31
m1: 2026-02-28
m2: 2026-01-31
m3: 2025-12-31
m4: 2025-11-30


+----------+------+---------+---------+---------+---------+---------+----------+
|billing_id|acc_id|shpd_gb_0|shpd_gb_1|shpd_gb_2|shpd_gb_3|shpd_gb_4|is_traffic|
+----------+------+---------+---------+---------+---------+---------+----------+
+----------+------+---------+---------+---------+---------+---------+----------+



In [22]:
fix_add_pay(spark, business_month_end).show(10)

dt_begin_fix_pay: 2025-10-31 00:00:00
m0: 2026-03-31
m1: 2026-02-28
m2: 2026-01-31
m3: 2025-12-31
m4: 2025-11-30


[Stage 17:====================================================>   (15 + 1) / 16]

+----------+------+--------+--------+--------+--------+--------+------+--------------+--------------+--------------+--------------+--------------+
|billing_id|acc_id|pay_00mm|pay_01mm|pay_02mm|pay_03mm|pay_04mm|is_pay|pay_class_00mm|pay_class_01mm|pay_class_02mm|pay_class_03mm|pay_class_04mm|
+----------+------+--------+--------+--------+--------+--------+------+--------------+--------------+--------------+--------------+--------------+
|         0|  2395|    0.00|  887.46|  583.10|  759.05|  985.29|     1|             0|             1|             4|             0|             4|
|         0| 51836|    0.00|  835.32|  597.30|  866.27|  591.94|     1|             0|             2|             2|             4|             1|
|         0| 81945|    0.00|  763.02|  590.46|  926.17|  571.13|     1|             0|             0|             0|             2|             4|
|         0| 88649|    0.00|  826.03|  807.96|  836.34|  891.14|     1|             0|             1|             1|  

In [18]:
spark.table('mtsfix_dm.agg_mtsfix_pay_fixa_a')

DataFrame[billing_id: int, acc_id: int, dt: date, amount: decimal(10,2), pay_class: int]

In [16]:
fix_add_att(spark, business_month_end).show(10)

+----------+------+----------+-----------+-----------+-----------+-----------+------------+----------+----------------------+-----------------+----------------+------------------+----------------+----------------+-----------------+--------------+--------------+--------------+--------------+---------------+
|billing_id|acc_id|   dt_open|dt_open_spd|dt_open_ctv|dt_open_atv|dt_open_tlf|dt_open_ictv|is_acc_att|is_contract_closed_flg|live_dur_contract|live_dur_fix_shd|live_dur_fix_telef|live_dur_fix_atv|live_dur_fix_ctv|live_dur_fix_ictv|has_opened_spd|has_opened_ctv|has_opened_atv|has_opened_tlf|has_opened_ictv|
+----------+------+----------+-----------+-----------+-----------+-----------+------------+----------+----------------------+-----------------+----------------+------------------+----------------+----------------+-----------------+--------------+--------------+--------------+--------------+---------------+
|        14|480253|2022-07-12| 2022-07-12| 2022-07-12| 2022-07-12| 2022-07-1

In [12]:
fix_conv(spark, business_month_end).show(10)

26/03/01 11:16:34 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+----------+------+--------+------------+-----------------------+-------+------------+-----------------+--------+--------+-----------+------------+------------------+-----------+-------+------+-----+----------------+---------------------+----------+----------------+---------+------------------+---------------------+----------------+-------------------+------+-------------+------+----------+--------------------+
|billing_id|acc_id|foris_id|foris_acc_id|personal_account_number|acc_num|foris_msisdn|foris_tariff_plan|is_foris|bopos_id|contract_id|contract_num|siebel_contract_id|customer_id|city_id|reg_id|mr_id|customer_type_id|calculation_method_id|   dt_from|is_noncommercial|is_closed|parent_contract_id|marketing_category_id|siebel_status_id|service_provider_id|is_b2b|orders_credit|credit|is_archive|customer_category_id|
+----------+------+--------+------------+-----------------------+-------+------------+-----------------+--------+--------+-----------+------------+------------------+----